In [21]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

# for student
train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

train_ids = list(set(train_labels.values))
val_ids = list(set(val_labels.values))

print(len(train_ids), train["battery_id"].nunique())
print(set(train["battery_id"].unique()) - set(train_ids))      # ids in train but not in train_ids
print(train.groupby("battery_id").size().max())                # longest battery, compare with 1497

140 140
set()
1497


In [22]:
features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
    "internal_resistance_mOhm",
    "cumulative_high_temp_hours",
]
features_B = features_A + ["cycle_index"]
features_C = features_B + ["ambient_temp_c"]
features = [features_A, features_B, features_C] # all feature sets together
label_names = ["A", "B", "C"]

In [23]:
x = []
for bid, g in df2.groupby("battery_id"):
    x.append(len(g))
print(min(x))
print(max(x))
print(np.mean(x))

95
1497
863.895


In [ ]:
def build_teacher_set(features):
    features = features + ["battery_id"]
    max_len = 1497
    # x: (window_len, num_training_samples, num_features)
    x = np.empty((max_len, len(train_ids), len(features)), dtype=float)
    for i, (bid, g) in enumerate(train[features].groupby("battery_id")):
        print(x.shape, g.values.shape)
        x[:len(g), i, :] = g.values

    return x

build_teacher_set(features_C)


(1497, 140, 7) (1023, 7)
(1497, 140, 7) (733, 7)
(1497, 140, 7) (962, 7)
(1497, 140, 7) (1075, 7)
(1497, 140, 7) (1009, 7)
(1497, 140, 7) (920, 7)
(1497, 140, 7) (717, 7)
(1497, 140, 7) (320, 7)
(1497, 140, 7) (1496, 7)
(1497, 140, 7) (1424, 7)
(1497, 140, 7) (1497, 7)
(1497, 140, 7) (442, 7)
(1497, 140, 7) (1360, 7)
(1497, 140, 7) (722, 7)
(1497, 140, 7) (403, 7)
(1497, 140, 7) (964, 7)
(1497, 140, 7) (582, 7)
(1497, 140, 7) (957, 7)
(1497, 140, 7) (1012, 7)
(1497, 140, 7) (1148, 7)
(1497, 140, 7) (565, 7)
(1497, 140, 7) (492, 7)
(1497, 140, 7) (1322, 7)
(1497, 140, 7) (847, 7)
(1497, 140, 7) (902, 7)
(1497, 140, 7) (552, 7)
(1497, 140, 7) (976, 7)
(1497, 140, 7) (1491, 7)
(1497, 140, 7) (408, 7)
(1497, 140, 7) (810, 7)
(1497, 140, 7) (1492, 7)
(1497, 140, 7) (1488, 7)
(1497, 140, 7) (1378, 7)
(1497, 140, 7) (433, 7)
(1497, 140, 7) (438, 7)
(1497, 140, 7) (1217, 7)
(1497, 140, 7) (1493, 7)
(1497, 140, 7) (431, 7)
(1497, 140, 7) (851, 7)
(1497, 140, 7) (861, 7)
(1497, 140, 7) (330, 7)


In [ ]:
from tensorflow.keras.utils import pad_sequences      # older versions: tensorflow.keras.preprocessing.sequence

class_labels = ["Normal", "Moderate", "Severe", "Critical"]
label_map = {c: i for i, c in enumerate(class_labels)}
assert set(train["degradation_status"].unique()) <= set(class_labels)   # catches naming mismatches

teacher_feats = features_C + ["time_since_previous_cycle_hours"]       # teacher may use extras the student can't
scaler_T = StandardScaler().fit(train[teacher_feats])                  # fit on train rows only

def build_sequences(df, feat_cols, scaler):
    X_list, y_list, idx_list, bids = [], [], [], []
    for bid, g in df.groupby("battery_id", sort=False):
        g = g.sort_values("cycle_index")
        A = scaler.transform(g[feat_cols]).astype("float32")
        A = np.hstack([A, np.ones((len(g), 1), "float32")])            # real-step indicator
        X_list.append(A)
        y_list.append(g["degradation_status"].map(label_map).to_numpy())
        idx_list.append(g.index.to_numpy())                            # original row labels
        bids.append(bid)
    X = pad_sequences(X_list, padding="post", dtype="float32", value=0.0)   # (B, T_max, F+1)
    y = pad_sequences(y_list, padding="post", value=-1)                      # (B, T_max), -1 = pad
    return X, y, idx_list, np.array(bids)

X_tr, y_tr, idx_tr, bids_tr = build_sequences(train, teacher_feats, scaler_T)
print(X_tr.shape, y_tr.shape)        # (140, 1497, len(teacher_feats)+1), (140, 1497)

(140, 1497, 8) (140, 1497)


In [ ]:
def build_teacher(n_feat, units=64, dropout=0.2, lr=1e-3):
    inp = keras.Input(shape=(None, n_feat+1))                 # None = any sequence length
    x = layers.Masking(mask_value=0.0)(inp)                 # skips padded steps (all-zero rows)
    x = layers.GRU(units, return_sequences=True)(x)         # unidirectional, so it is causal
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(4, activation="softmax")(x)          # prediction at every time step
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  weighted_metrics=["accuracy"])
    return model

def fit_teacher(teacher, X_train, y_train, X_val, y_val, epochs=100):
    w_train = (y_train != -1).astype("float32") # make into 1 or 0
    w_val = (y_val != -1).astype("float32") # make into 1 or 0
    y_train_fit = np.where(y_train == -1, 0, y_train)
    y_val_fit = np.where(y_val == -1, 0, y_val)

    teacher.fit(
        X_train, y_train_fit, sample_weight=w_train,
        validation_data=(X_val, y_val_fit, w_val),
        batch_size=16, epochs=epochs,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
        verbose=2,
    )

    return teacher

def build_student(n_feat, hidden=(64, 32), dropout=0.15, lr=1e-3, alpha=0.5, tau=2.0, eps=1e-7):

    def make_distill_loss(alpha, tau):
        # y_true: (batch, 8) = [one-hot hard label (4) | teacher probabilities (4)]
        def loss(y_true, y_pred):
            y_hard = y_true[:, :4]
            p_t = tf.clip_by_value(y_true[:, 4:], eps, 1.0)
            p_s = tf.clip_by_value(y_pred, eps, 1.0)

            # hard-label term: ordinary cross-entropy at temperature 1
            ce = -tf.reduce_sum(y_hard * tf.math.log(p_s), axis=-1)

            # soft-label term: KL(teacher || student) at temperature tau.
            # softmax(log(p)/tau) equals softmax(logits/tau), so no access to logits is needed.
            log_s = tf.nn.log_softmax(tf.math.log(p_s) / tau, axis=-1)
            log_t = tf.nn.log_softmax(tf.math.log(p_t) / tau, axis=-1)
            kl = tf.reduce_sum(tf.exp(log_t) * (log_t - log_s), axis=-1) * tau**2

            return alpha * ce + (1.0 - alpha) * kl
        return loss
    
    def hard_accuracy(y_true, y_pred):
        # accuracy against the hard label, which is the first 4 columns of the combined target
        return tf.keras.metrics.categorical_accuracy(y_true[:, :4], y_pred)

    inp = keras.Input(shape=(n_feat,), name="num")
    x = inp
    for h in hidden:
        x = layers.Dense(h, use_bias=False)(x) # BN supplies the bias
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(4, activation="softmax", name="status")(x) # still plain probabilities at inference
    student = keras.Model(inp, out)

    student.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=make_distill_loss(alpha, tau, eps),
                  metrics=[hard_accuracy])
    return student

def make_targets(y_int, p_soft):
    # y_int: (N,) integer labels 0..3,  p_soft: (N, 4) probabilities -> (N, 8)
    return np.hstack([np.eye(4)[y_int], p_soft]).astype("float32")

def fit_student(student, X_train, y_train, p_teacher_train, X_val, y_val,
                hidden, dropout_rate, learning_rate, unit, alpha, tau,
                epochs=200, use_class_weights=True):

    # training targets: [one-hot hard label | out-of-fold teacher probabilities]
    Y_train = make_targets(y_train, p_teacher_train)
    # validation targets: the teacher is not involved, so use the one-hot label in both halves
    Y_val = make_targets(y_val, np.eye(4)[y_val])

    # optional class weighting: one weight per row, based on its true class
    w_train = None
    if use_class_weights:
        cw = compute_class_weight("balanced", classes=np.arange(4), y=y_train)
        w_train = cw[y_train].astype("float32")

    student.fit(
        X_train, Y_train, sample_weight=w_train,
        validation_data=(X_val, Y_val),
        batch_size=256, epochs=epochs,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_hard_ce", patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_hard_ce", factor=0.5, patience=5),
        ],
        verbose=2,
    )

    p = student.predict(X_val, batch_size=1024) # (N_val, 4) probabilities
    pred = p.argmax(-1) # (N_val,) class indices

    metrics = {
        "hidden": hidden,
        "dropout_rate": dropout_rate,
        "learning_rate": learning_rate,
        "alpha": alpha,
        "tau": tau,
        "unit": unit,
        "accuracy": accuracy_score(y_val, pred),
        "macro_F1": f1_score(y_val, pred, average="macro", zero_division=0),
        "val_probs": p,
        "val_pred": pred
    }
    metrics["val_error"] = 1 - metrics["accuracy"]
    return student, metrics

def build_sequences(df, features):

    preprocess = Pipeline([
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold(threshold=0.0))
    ])

    X_list, y_list, idx_list, bids = [], [], [], []
    for bid, g in df.groupby("battery_id", sort=False):
        g = g.sort_values("cycle_index")
        A = preprocess.fit_transform(g[features]).astype("float32")
        A = np.hstack([A, np.ones((len(g), 1), "float32")])            # real-step indicator
        X_list.append(A)
        y_list.append(g["degradation_status"].map(label_map).to_numpy())
        idx_list.append(g.index.to_numpy())                            # original row labels
        bids.append(bid)
    X = pad_sequences(X_list, padding="post", dtype="float32", value=0.0)   # (B, T_max, F+1)
    y = pad_sequences(y_list, padding="post", value=-1)                      # (B, T_max), -1 = pad
    return X, y, idx_list

def teacher_row_probs(teacher, X_seq, idx_list, row_index):
    """
    converts the teacher's output (ie. probability structure), which is a 3D structure,
    to a 2D structure such that it is usable as the target for the student.
    idx_list: is list of battery_ids, same order as X_train_teacher.
    """
    p = teacher.predict(X_seq, batch_size=16) # (B, T_max, 4)
    out = pd.DataFrame(np.nan, index=row_index, columns=range(4))
    for i, rows in enumerate(idx_list): # rows = list of cycle indices of battery i
        out.loc[rows, :] = p[i, :len(rows), :] # real steps only
    assert not out.isna().any().any()
    return out.loc[row_index].to_numpy().astype("float32")

In [ ]:
from itertools import product
import seaborn as sns
from pathlib import Path

class_names = ["Normal", "Moderate", "Severe", "Critical"]
label_map = {c: i for i, c in enumerate(class_labels)}

DROPOUT_RATES = [0.0, 0.15]
LEARNING_RATES = [1e-2, 1e-5]
UNITS = [32, 64]
HIDDEN = [
    (128, 64),
    (128, 64, 32)
]

units_mapping = {UNITS[x]:x for x in range(len(UNITS))}
dropout_rate_mapping = {DROPOUT_RATES[x]:x for x in range(len(DROPOUT_RATES))}
learning_rate_mapping = {LEARNING_RATES[x]:x for x in range(len(LEARNING_RATES))}
hidden_mapping = {HIDDEN[x]:x for x in range(len(HIDDEN))}

optimize_metric = "ARI" # out of ARI, silhouette, NMI, accuracy, macro_F1

l1,l2,l3,l4 = len(UNITS),len(DROPOUT_RATES),len(UNITS),len(HIDDEN)
alpha = 0.5
tau = 2

def run_for_features(use_features, label):

    val_pred_struct = np.empty((l1,l2,l3,l4), dtype=object)
    all_metrics = []

    # build teacher input structure
    
    X_train_teacher, y_train_teacher, idx_list_train = build_sequences(train, use_features)
    X_val_teacher, y_val_teacher, idx_list_val = build_sequences(val, use_features)

    preprocess = Pipeline([
        ("scaler", StandardScaler()),
        ("variance", VarianceThreshold(threshold=0.0))
    ])

    X_train_student = preprocess.fit_transform(train[use_features])
    X_val_student = preprocess.transform(val[use_features])
    y_train_student = train["degradation_status"].to_numpy()
    y_val_student = val["degradation_status"].to_numpy()

    for i, j, k, l in product(
        range(len(UNITS)),
        range(len(DROPOUT_RATES)),
        range(len(LEARNING_RATES)),
        range(len(HIDDEN))
    ):

        units = UNITS[i]
        dropout_rate = DROPOUT_RATES[j]
        learning_rate = LEARNING_RATES[k]
        hidden = HIDDEN[l]

        teacher = build_teacher(len(use_features), units, dropout_rate, learning_rate)
        teacher = fit_teacher(teacher, X_train_teacher, y_train_teacher, X_val_teacher, y_val_teacher)
        student = build_student(len(use_features), hidden, dropout_rate, learning_rate, alpha, tau)
        p_teacher_train = teacher_row_probs(teacher, X_train_teacher, idx_list_train, train.index)
        student, metrics = fit_student(student, X_train_student, y_train_student, p_teacher_train, X_val_student, y_val_student, hidden, dropout_rate, learning_rate, units, alpha, tau)
        all_metrics.append(metrics)
        val_pred_struct[i,j,j] = metrics["val_pred"]


    metrics_df = pd.DataFrame(all_metrics).sort_values(optimize_metric, ascending=False)
    best = metrics_df.iloc[0]
    best_pred = val_pred_struct[
        units_mapping[metrics_df["unit"].values[0]],
        dropout_rate_mapping[metrics_df["dropout_rate"].values[0]],
        learning_rate_mapping[metrics_df["learning_rate"].values[0]],
        hidden_mapping[metrics_df["hidden"].values[0]],
    ]

    cm = confusion_matrix(y_val_student, best_pred, labels=class_names)
    cm_frac = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)   # correct / total per actual class
    annot = np.array([[f"{cm_frac[i, j]:.0%}\n({cm[i, j]})" for j in range(cm.shape[1])]
                        for i in range(cm.shape[0])])

    fig, ax = plt.subplots(2, 2, figsize=(20, 5), layout="constrained")
    x1, x2 = ax
    ax_cm, ax_err_units = x1
    ax_err_dropout, ax_err_learning = x2

    sns.heatmap(cm_frac, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1,
                xticklabels=[f"Predicted {c}" for c in class_names],
                yticklabels=[f"Actual {c}" for c in class_names],
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "fraction of the actual class"}, ax=ax_cm)
    ax_cm.set_title(f"Validation confusion matrix\n"
                    f"(units used: {best['units']}, hidden structure used: {best['hidden']}, dropout_rate used: {best['dropout_rate']}, learning_rate used: {best['learning_rate']} best by {optimize_metric})")
    ax_cm.tick_params(axis="x", rotation=30)
    ax_cm.tick_params(axis="y", rotation=0)

    # validation error of every run
    for u, g in metrics_df.groupby("units"):
        ax_err_units.plot(g["units"], g["val_error"], marker="o", label=u, linestyle="none")
    ax_err_units.scatter([best["units"]], [best["val_error"]], marker="*", s=250,
                    color="black", zorder=5, label=f"best by {optimize_metric}")
    for dropout, g in metrics_df.groupby("dropout_rate"):
        ax_err_dropout.plot(g["dropout_rate"], g["val_error"], marker="o", label=dropout, linestyle="none")
    ax_err_dropout.scatter([best["dropout_rate"]], [best["val_error"]], marker="*", s=250,
                    color="black", zorder=5, label=f"best by {optimize_metric}")
    for lr, g in metrics_df.groupby("learning_rate"):
        ax_err_learning.plot(g["learning_rate"], g["val_error"], marker="o", label=lr, linestyle="none")
    ax_err_learning.scatter([best["learning_rate"]], [best["val_error"]], marker="*", s=250,
                    color="black", zorder=5, label=f"best by {optimize_metric}")
    
    ax_err_learning.set_xscale("log")
    ax_err_units.set_xscale("log")
    ax_err_dropout.set_xscale("log")

    ax_err_learning.set_xlabel("latent_dim")
    ax_err_units.set_xlabel("hidden_dim")
    ax_err_dropout.set_xlabel("dropout_rate")

    ax_err_learning.set_ylabel("validation error (1 - accuracy)")
    ax_err_units.set_ylabel("validation error (1 - accuracy)")
    ax_err_dropout.set_ylabel("validation error (1 - accuracy)")

    ax_err_learning.set_title("Validation error per run")
    ax_err_units.set_title("Validation error per run")
    ax_err_dropout.set_title("Validation error per run")

    ax_err_learning.grid(alpha=0.2)
    ax_err_units.grid(alpha=0.2)
    ax_err_dropout.grid(alpha=0.2)

    ax_err_learning.legend()
    ax_err_units.legend()
    ax_err_dropout.legend()
    
        
    fig.suptitle(f"Label: {label}", fontsize=12)
    plt.show()
    plt.savefig(Path.cwd() / "images" / "supervised_teacher_student" / f"set_{label}.png")

    return metrics_df

all_metrics = {}
for label, feature in zip(label_names, features):
    metrics_df = run_for_features(feature, label)
    print('--label--', label, 'metric df:'); print(metrics_df); print()
